In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# QESEM: Isang Qiskit Function ng Qedma
*Tingnan ang [API reference](https://docs.quantum.ibm.com/api/functions/qedma-qesem)*

> **Note:** Ang Qiskit Functions ay isang experimental na feature na available lamang sa mga gumagamit ng IBM Quantum&reg; Premium Plan, Flex Plan, at On-Prem (sa pamamagitan ng IBM Quantum Platform API) Plan. Nasa preview release status pa ito at maaaring magbago.

## Pangkalahatang-ideya
Kahit na malaki na ang pagbabago ng mga quantum processing unit sa mga nakaraang taon, ang mga error dahil sa ingay at mga kakulangan sa kasalukuyang hardware ay nananatiling pangunahing hamon para sa mga developer ng quantum algorithm. Habang papalapit ang larangan sa mga utility-scale na quantum computation na hindi na mapapatunayan nang klasikal, nagiging mas mahalaga ang mga solusyon para sa pagkansela ng ingay na may garantisadong katumpakan. Para malampasan ang hamong ito, binuo ng Qedma ang Quantum Error Mitigation (QESEM), na seamlessly na naka-integrate sa IBM Quantum Platform bilang isang [Qiskit Function](/guides/functions).

Gamit ang QESEM, maaaring patakbuhin ng mga gumagamit ang kanilang mga quantum circuit sa maingay na QPU para makakuha ng lubos na tumpak na mga resulta na walang error, na may napaka-episyenteng gastos sa QPU time, malapit sa mga pundamental na hangganan. Para magawa ito, ginagamit ng QESEM ang isang hanay ng mga proprietary na pamamaraan na binuo ng Qedma para sa pag-characterize at pagbabawas ng mga error. Kasama sa mga pamamaraan ng pagbabawas ng error ang gate optimization, noise-aware transpilation, error suppression (ES), at unbiased error mitigation (EM). Sa kombinasyong ito ng mga characterization-based na pamamaraan, nakakamit ng mga gumagamit ang maaasahan at error-free na mga resulta para sa mga generic na large-volume quantum circuit, na nagbubukas ng mga application na hindi maisasagawa nang iba.

Para sa buong paglalarawan ng mga pinagbabatayan na bahagi, pati na rin ang isang utility-scale na demonstrasyon, tingnan ang papel na [Reliable high-accuracy error mitigation for utility-scale quantum circuits](https://arxiv.org/abs/2508.10997).
## Paglalarawan
Maaari mong gamitin ang QESEM function ng Qedma para madaling i-estimate at i-execute ang iyong mga circuit na may error suppression at mitigation, na nakakamit ng mas malalaking circuit volume at mas mataas na katumpakan. Para gamitin ang QESEM, magbibigay ka ng isang quantum circuit, isang hanay ng mga observable na susukat, isang target na statistical accuracy para sa bawat observable, at isang piniling QPU. Bago patakbuhin ang circuit sa target na katumpakan, maaari mong i-estimate ang kinakailangang QPU time batay sa isang analytical na kalkulasyon na hindi nangangailangan ng circuit execution. Kapag nasiyahan ka na sa QPU time estimation, maaari mo nang i-execute ang circuit gamit ang QESEM.

Kapag nag-execute ka ng circuit, nagpapatakbo ang QESEM ng device characterization protocol na akma sa iyong circuit, na nagbubunga ng maaasahang noise model para sa mga error na nangyayari sa circuit. Batay sa characterization, unang ipinapatupad ng QESEM ang noise-aware transpilation para i-map ang input circuit sa isang hanay ng mga physical qubit at gate, na nagpapababa ng ingay na nakakaapekto sa target na observable. Kasama dito ang mga natively available na gate (CX/CZ sa IBM&reg; na mga device), pati na rin ang mga karagdagang gate na na-optimize ng QESEM, na bumubuo ng extended gate set ng QESEM. Nagpapatakbo rin ang QESEM ng isang hanay ng mga characterization-based na ES at EM circuit sa QPU at kinokolekta ang kanilang mga resulta ng pagsukat. Pagkatapos ay klasikal na pinoproseso ang mga ito para magbigay ng unbiased na expectation value at error bar para sa bawat observable, na naaayon sa hiniling na katumpakan.

![Pangkalahatang-ideya ng Qedma QESEM](../docs/images/guides/qedma-qesem/overview.svg)
Napatunayang nagbibigay ang QESEM ng mga resulta na may mataas na katumpakan para sa iba't ibang quantum application at sa pinakamalaking circuit volume na makakamit ngayon. Nag-aalok ang QESEM ng mga sumusunod na feature para sa mga gumagamit, na ipinapakita sa seksyon ng mga benchmark sa ibaba:
-	**Garantisadong katumpakan:** Naglalabas ang QESEM ng mga unbiased na pagtatantya para sa mga expectation value ng mga observable. Ang EM method nito ay may mga theoretical na garantiya na - kasama ang cutting-edge na characterization ng Qedma - tinitiyak na ang mitigation ay nagko-converge sa noiseless na output ng circuit hanggang sa user-specified na katumpakan. Kumpara sa maraming heuristic na EM method na madaling magkaroon ng sistematikong error o bias, ang garantisadong katumpakan ng QESEM ay mahalaga para matiyak ang maaasahang mga resulta sa mga generic na quantum circuit at observable.
-	**Scalability sa malalaking QPU:** Ang QPU time ng QESEM ay depende sa circuit volume, ngunit independyente sa bilang ng mga qubit. Naipakita na ng Qedma ang QESEM sa pinakamalaking quantum device available ngayon, kabilang ang IBM Quantum 127-qubit Eagle at 133-qubit Heron na mga device.
-	**Application-agnostic:** Napatunayan na ang QESEM sa iba't ibang application, kabilang ang Hamiltonian simulation, VQE, QAOA, at amplitude estimation. Maaaring mag-input ang mga gumagamit ng anumang quantum circuit at observable na susukat, at makakuha ng tumpak na mga resulta na walang error. Ang mga limitasyon lamang ay idinidikta ng mga hardware specification at inilaan na QPU time, na nagtatakda ng naa-access na circuit volume at output accuracy. Kumpara rito, maraming solusyon sa pagbabawas ng error ang application-specific o gumagamit ng hindi kontroladong heuristic, na ginagawa silang hindi angkop para sa mga generic na quantum circuit at application.
-  **Extended gate set:** Sinusuportahan ng QESEM ang mga fractional-angle gate, at nagbibigay ng Qedma-optimized fractional-angle $Rzz(\theta)$ gate sa mga IBM Quantum Heron at Eagle device. Ang extended gate set na ito ay nagpapahintulot ng mas episyenteng compilation at nagbubukas ng mga circuit volume na mas malaki ng hanggang 2x kumpara sa default na CX/CZ compilation.
-	**Multibase observables:** Sinusuportahan ng QESEM ang mga input observable na binubuo ng maraming non-commuting na Pauli string, tulad ng mga generic na Hamiltonian. Ang pagpili ng mga measurement basis at ang optimization ng QPU resource allocation (shots at circuits) ay awtomatikong ginagawa ng QESEM para maibaba ang kinakailangang QPU time para sa hiniling na katumpakan. Ang optimization na ito, na isinasaalang-alang ang hardware fidelity at execution rate, ay nagpapahintulot sa iyo na magpatakbo ng mas malalim na circuit at makamit ang mas mataas na katumpakan.
## Mga Benchmark
Nasubukan na ang QESEM sa malawak na iba't ibang kaso ng paggamit at application. Makakatulong ang mga sumusunod na halimbawa sa iyo sa pag-assess kung anong uri ng workload ang maaari mong patakbuhin gamit ang QESEM.

Isang mahalagang sukatan para ma-quantify ang kahirapan ng parehong error mitigation at classical simulation para sa isang partikular na circuit at observable ay ang **active volume**: ang bilang ng mga CNOT gate na nakakaapekto sa observable sa circuit. Ang active volume ay depende sa depth at width ng circuit, sa timbang ng observable, at sa estruktura ng circuit, na nagtatakda ng light cone ng observable. Para sa karagdagang detalye, tingnan ang talumpati mula sa [2024 IBM Quantum Summit](https://www.youtube.com/watch?v=Hd-IGvuARfE&t=1730s). Nagbibigay ang QESEM ng partikular na malaking halaga sa high-volume regime, na nagbibigay ng maaasahang mga resulta para sa mga generic na circuit at observable.

![Active volume](../docs/images/guides/qedma-qesem/active_volume.svg)

| Application    | Bilang ng qubit | Device | Paglalarawan ng circuit | Katumpakan | Kabuuang oras | Paggamit ng runtime |
| ---------  | ---------------- | ----- | -------------------------- | -------- | ---------- | ------------- |
| VQE circuit  | 8              | Eagle (r3) | 21 total layers, 9 measurement bases, 1D chain                    | 98%      | 35 min      | 14 min         |
| Kicked Ising   | 28              | Eagle (r3) | 3 unique layers x 3 steps, 2D heavy-hex topology                      | 97%     | 22 min      | 4 min          |
| Kicked Ising   | 28              | Eagle (r3) | 3 unique layers x 8 steps, 2D heavy-hex topology                     | 97%      | 116 min      | 23 min          |
| Trotterized Hamiltonian simulation   | 40  | Eagle (r3)            | 2 unique layers x 10 Trotter steps, 1D chain                    | 97%      | 3 hours     | 25 min         |
| Trotterized Hamiltonian simulation   | 119  | Eagle (r3)           | 3 unique layers x 9 Trotter steps, 2D heavy-hex topology                    | 95%      | 6.5 hours     | 45 min         |
| Kicked Ising  | 136             | Heron (r2) | 3 unique layers x 15 steps, 2D heavy-hex topology                    | 99%      | 52 min             | 9 min           |

Ang katumpakan dito ay sinusukat kumpara sa ideal na halaga ng observable: $\frac{\langle O \rangle_{ideal} - \epsilon}{\langle O \rangle_{ideal}}$, kung saan ang '$\epsilon$' ay ang absolute na katumpakan ng mitigation (itinakda ng input ng gumagamit), at ang $\langle O \rangle_{ideal}$ ay ang observable sa noiseless na circuit.
Ang 'Runtime usage' ay sumusukat ng paggamit ng benchmark sa batch mode (kabuuan ng paggamit ng mga indibidwal na job), habang ang 'total time' ay sumusukat ng paggamit sa session mode (experiment wall time), na kinabibilangan ng karagdagang classical at communication time. Available ang QESEM para sa execution sa parehong mode, para magamit ng mga gumagamit ang kanilang mga available na mapagkukunan nang pinakamabuti.

Ang mga 28-qubit Kicked Ising circuit ay nagsi-simulate ng Discrete Time Quasicrystal na pinag-aralan ni Shinjo et al. (tingnan ang [arXiv 2403.16718](https://arxiv.org/abs/2403.16718) at [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)) sa tatlong magkakaugnay na loop ng ibm_kawasaki. Ang mga parameter ng circuit na ginamit dito ay $(\theta_x, \theta_z) = (0.9 \pi, 0)$, na may ferromagnetic na paunang estado $| \psi_0 \rangle = | 0 \rangle ^{\otimes n}$. Ang sinusukat na observable ay ang absolute na halaga ng magnetization $M = |\frac{1}{28} \sum_{i=0}^{27} \langle Z_i \rangle|$. Ang utility-scale na Kicked Ising experiment ay pinatakbo sa 136 pinakamahusay na qubit ng ibm_fez; ang partikular na benchmark na ito ay pinatakbo sa Clifford angle na $(\theta_x, \theta_z) = (\pi, 0)$, kung saan ang active volume ay mabagal na lumalaki kasabay ng circuit depth, na — kasama ang mataas na fidelity ng device — nagbibigay-daan sa mataas na katumpakan sa maikling runtime.

Ang mga Trotterized Hamiltonian simulation circuit ay para sa isang Transverse-Field Ising model sa mga fractional angle: $(\theta_{zz}, \theta_x) = (\pi / 4, \pi /8)$ at $(\theta_{zz}, \theta_x) = (\pi / 6, \pi / 8)$ ayon sa pagkakasunod (tingnan ang [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)). Ang utility-scale na circuit ay pinatakbo sa 119 pinakamahusay na qubit ng ibm_brisbane, habang ang 40-qubit na eksperimento ay pinatakbo sa pinakamahusay na available na chain. Ang katumpakan ay iniulat para sa magnetization; nakamit din ang mga resulta na may mataas na katumpakan para sa mga observable na may mas mataas na timbang.

Ang VQE circuit ay binuo kasama ang mga mananaliksik mula sa Center for Quantum Technology and Applications sa Deutsches Elektronen-Synchrotron (DESY). Ang target na observable dito ay isang Hamiltonian na binubuo ng malaking bilang ng mga non-commuting na Pauli string, na binibigyang-diin ang optimized na performance ng QESEM para sa multi-basis observable. Ang mitigation ay inilapat sa isang classically-optimized na ansatz; kahit hindi pa nailalathala ang mga resultang ito, makukuha ang parehong kalidad ng mga resulta para sa iba't ibang circuit na may magkaparehong istruktura.
## Pagsisimula
Mag-authenticate gamit ang iyong [IBM Quantum Platform API key](http://quantum.cloud.ibm.com/), at piliin ang QESEM Qiskit Function tulad ng sumusunod. (Ipinapalagay ng snippet na ito na [na-save mo na ang iyong account](/guides/functions#install-qiskit-functions-catalog-client) sa iyong lokal na environment.)

In [1]:
import qiskit

from qiskit_ibm_catalog import QiskitFunctionsCatalog


catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [2]:
# load the function
qesem_function = catalog.load("qedma/qesem")

## Examples

To get started, try this basic example of estimating the required QPU time to run QESEM for a given `pub`:

In [3]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [4]:
circ = qiskit.QuantumCircuit(5)
circ.cx(0, 1)
circ.cx(2, 3)
circ.cx(1, 2)
circ.cx(3, 4)

avg_magnetization = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("Z", [q], 1 / 5) for q in range(5)], num_qubits=5
)
other_observable = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("ZZ", [0, 1], 1.0), ("XZ", [1, 4], 0.5)], num_qubits=5
)

time_estimation_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    options={
        "estimate_time_only": "analytical",
    },
    backend_name=backend_name,  # E.g. "ibm_fez"
)

time_estimate_result = time_estimation_job.result()

Maaari mong gamitin ang mga pamilyar na Qiskit Serverless API para suriin ang status ng iyong Qiskit Function workload o makuha ang mga resulta:

In [5]:
sample_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    backend_name=backend_name,  # E.g. "ibm_fez"
)

Ang sumusunod na code snippet ay nagpapaliwanag kung paano kunin ang QPU time estimation (kapag itinakda ang `estimate_time_only`):

In [6]:
# Print the ID so you can use it later, if necessary
print(sample_job.job_id)

print(sample_job.status())
result = sample_job.result()

9a87a23f-82f5-429e-91fb-cc8a9d107980
QUEUED


Ang sumusunod na code snippet ay nagpapakita kung paano kunin ang mga resulta ng mitigation (kapag hindi itinakda ang `estimate_time_only`) at mga execution metric. Naglalaman ang mga ito ng mahahalagang data na nagpapahintulot sa mas malalim na pag-unawa kung paano nakakaapekto ang iba't ibang parameter sa QESEM execution. Maaari rin itong maging kaugnay kapag nagsusulat ng papel batay sa iyong pananaliksik.

In [7]:
print(
    f"The estimated QPU time for this PUB is: "
    f"\n{time_estimate_result[0].metadata}"
)

The estimated QPU time for this PUB is: 
{'time_estimation_sec': 1800, 'description': None, 'instance': 'crn:v1:bluemix:public:quantum-computing:us-east:a/6c63dae5281147f1a0449b36e0aaba3a:ae40ab55-8c55-4042-9204-71a6541d56ec::', 'transpilation_level': 'standard', 'parallel_execution': False, 'total_qpu_time': 0, 'empirical_estimation_mitigation_results': None, 'resource_usage': {'RUNNING: MAPPING': {'CPU_TIME': 42.44837867887691, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: OPTIMIZING_FOR_HARDWARE': {'CPU_TIME': 17.879877626895905, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: WAITING_FOR_QPU': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: EXECUTING_QPU': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}}}


## Kunin ang mga mensahe ng error
Kung ang status ng iyong workload ay ERROR, gamitin ang `job.result()` para kunin ang mensahe ng error tulad ng sumusunod:

In [9]:
results = result[0]
print(f"Mitigated expectation values: \n{results.data.evs}")
print(f"Mitigated error-bar: \n{results.data.stds}")
noisy_results = results.metadata["noisy_results"]
print(f"Noisy expectation values: \n{noisy_results.evs}")
print(f"Noisy error-bar: \n{noisy_results.stds}")
print(f"Total QPU time: \n {results.metadata['total_qpu_time']}")
print(
    f"Gates fidelity measured during the experiment: "
    f"\n {results.metadata['gate_fidelities']}"
)
print(
    f"Total shots / mitigation shots: \n "
    f"{results.metadata['total_shots']} / "
    f"{results.metadata['mitigation_shots']}"
)
print("Transpiled circuits:")
for i, circuit in enumerate(results.metadata["transpiled_circs"]):
    print(f"Circuit {i}:")
    print(f"  Circuit: \n {circuit['circuit']}")
    if "qubit_map" in circuit:
        print(f"  Qubit mapping: \n {circuit['qubit_map']}")
    print(f"  Measurement bases: \n {circuit['num_measurement_bases']}")

Mitigated expectation values: 
[1.00169764 1.00460812]
Mitigated error-bar: 
[0.00155021 0.0099558 ]
Noisy expectation values: 
[0.95717143 0.94271429]
Noisy error-bar: 
[0.00206982 0.00872689]
Total QPU time: 
 150.0
Gates fidelity measured during the experiment: 
 {'CZ': 0.9979051606662408, 'ID1Q': 0.9993865847216725}
Total shots / mitigation shots: 
 495600 / 220400
Transpiled circuits:
Circuit 0:
  Circuit: 
 OPENQASM 3.0;
include "stdgates.inc";
bit[145] c0;
qubit[145] q0;
rz(-pi) q0[143];
rz(0) q0[141];
rz(-pi) q0[140];
sx q0[143];
sx q0[141];
sx q0[140];
rz(-pi/2) q0[143];
rz(pi/2) q0[141];
rz(-pi/2) q0[140];
sx q0[143];
sx q0[141];
sx q0[140];
rz(pi/2) q0[143];
rz(-pi/2) q0[141];
rz(pi/2) q0[140];
barrier q0[140], q0[141], q0[142], q0[143], q0[144];
cz q0[144], q0[143];
cz q0[142], q0[141];
barrier q0[144], q0[143], q0[142], q0[141];
barrier q0[140], q0[141], q0[142], q0[143], q0[144];
rz(-pi) q0[144];
rz(-pi/2) q0[143];
rz(0) q0[142];
rz(-pi/2) q0[141];
sx q0[144];
sx q0[143];

## Fetch error messages

If your workload status is ERROR, use `job.result()` to fetch the error message to fetch the error message as follows:

In [14]:
# Get the result and truncate for readability
result = sample_job.result()
result_str = str(result)
max_length = 500  # Adjust this value as necessary

if len(result_str) > max_length:
    truncated = (
        result_str[:max_length]
        + f"... (truncated {len(result_str) - max_length} characters)"
    )
else:
    truncated = result_str

print(truncated)

PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(2,), dtype=float64>), stds=np.ndarray(<shape=(2,), dtype=float64>), shape=(2,)), metadata={'gate_fidelities': {'CZ': 0.9979051606662408, 'ID1Q': 0.9993865847216725}, 'total_shots': 495600, 'mitigation_shots': 220400, 'transpiled_circs': [{'circuit': 'OPENQASM 3.0;\ninclude "stdgates.inc";\nbit[145] c0;\nqubit[145] q0;\nrz(-pi) q0[143];\nrz(0) q0[141];\nrz(-pi) q0[140];\nsx q0[143];\nsx q0[141];\nsx q0[140];\nrz(-pi/2) q0[143];\nrz(pi... (truncated 4174 characters)


## Makakuha ng suporta

Nandito ang support team ng Qedma para tumulong! Kung nakatagpo ka ng anumang isyu o mayroon kang mga tanong tungkol sa paggamit ng QESEM Qiskit Function, huwag mag-atubiling makipag-ugnayan. Ang aming mga kaalamang at magiliw na support staff ay handang tumulong sa iyo sa anumang teknikal na alalahanin o katanungan na maaaring mayroon ka.

Maaari kang mag-email sa amin sa support@qedma.com para sa tulong. Pakisama ang pinakamaraming detalye hangga't maaari tungkol sa isyung nararanasan mo para matulungan kaming magbigay ng mabilis at tumpak na tugon. Maaari ka ring makipag-ugnayan sa iyong dedikadong Qedma POC representative sa pamamagitan ng email o telepono.

Para matulungan kaming tulungan ka nang mas episyente, pakibigay ang sumusunod na impormasyon kapag nakipag-ugnayan ka sa amin:

- Isang detalyadong paglalarawan ng isyu
- Ang job ID
- Anumang kaugnay na mensahe ng error o code

Nakatuon kami sa pagbibigay sa iyo ng mabilis at epektibong suporta para matiyak na mayroon kang pinakamahusay na karanasan sa aming Qiskit Function.

Palagi kaming naghahanap ng paraan para mapabuti ang aming produkto at tinatanggap namin ang iyong mga mungkahi! Kung mayroon kang mga ideya kung paano namin mapapahusay ang aming mga serbisyo o mga feature na nais mong makita, pakipadala ang iyong mga kaisipan sa support@qedma.com.

## Mga susunod na hakbang

> **Tip:** - [Humiling ng access sa Qedma QESEM](https://quantum.cloud.ibm.com/functions?id=qedma-qesem).
> - Bisitahin ang [API reference](https://docs.quantum.ibm.com/api/functions/qedma-qesem) para sa Qiskit Function na ito.
> - Suriin ang [Bauman, N. P., et al. (2025). Coupled Cluster Downfolding Theory in Simulations of Chemical Systems on Quantum Hardware. arXiv preprint arXiv:2507.01199](https://arxiv.org/pdf/2507.01199).
> - Suriin ang [Aharonov, D., et al. (2025). Reliable high-accuracy error mitigation for utility-scale quantum circuits. arXiv preprint arXiv:2508.10997](https://arxiv.org/pdf/2508.10997).